# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step template for loading, exploring, and analyzing the [FAIR^2 dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

Set the Croissant schema URL below:

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print high-level dataset information
print(f"{metadata.name}: {metadata.description}\n")
print(f"Published: {getattr(metadata, 'datePublished', 'Unknown')}")
print(f"Version: {getattr(metadata, 'version', 'Unknown')}")
print(f"License: {getattr(metadata, 'license', 'Unknown')}")

## 2. Data Overview
Review available record sets and fields. All entities are referenced by their `@id` fields following Croissant best practices.

In [ ]:
# Helper function: Print basic info for any object using @id, name, and description
def print_entity(entity, prefix=''):
    _id = getattr(entity, '@id', 'N/A')
    name = getattr(entity, 'name', 'N/A')
    desc = getattr(entity, 'description', 'N/A')
    print(f"{prefix}ID: {_id}\n{prefix}  Name: {name}\n{prefix}  Description: {desc}")

# Collect all record sets in the dataset metadata
record_sets = getattr(metadata, 'recordSet', [])
if not isinstance(record_sets, list):
    record_sets = [record_sets]
print(f"Number of record sets found: {len(record_sets)}\n")

for i, rs in enumerate(record_sets):
    print(f"--- Record set {i+1}: ---")
    print_entity(rs)
    # Fields in the record set:
    fields = getattr(rs, 'field', [])
    if not isinstance(fields, list):
        fields = [fields]
    for f in fields:
        print("  - Field @id:", getattr(f, '@id', 'N/A'))
        print("      name:", getattr(f, 'name', 'N/A'))
        print("      dataType:", getattr(f, 'dataType', 'N/A'))
    print()

## 3. Data Extraction
Load data from each record set into a Pandas DataFrame for analysis. 

Use the record set and field `@id`s from the overview above.

In [ ]:
# Prepare to extract data from all record sets
dataframes = {}
record_set_ids = [getattr(rs, '@id', None) for rs in record_sets if getattr(rs, '@id', None) is not None]

for rs in record_sets:
    rs_id = getattr(rs, '@id', None)
    if rs_id is None:
        continue
    print(f"Loading records for record set '@id': {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"  - Columns: {df.columns.tolist()}")
    print(f"  - Sample records:")
    display(df.head(2))

# For demonstration and EDA we'll use the first available record set
if record_set_ids:
    main_record_set_id = record_set_ids[0]
    print(f"\nMain record set id for EDA: {main_record_set_id}")
    print(f"Fields (columns):", dataframes[main_record_set_id].columns.tolist())
else:
    main_record_set_id = None

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps: filter numeric values, normalize, and group records. 

This example uses the first numeric field found in the chosen record set.

In [ ]:
import numpy as np

if main_record_set_id:
    df = dataframes[main_record_set_id]
    # Find the first numeric field (int/float)
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if not numeric_cols:
        # Try to convert possible numeric columns (sometimes loaded as object)
        for col in df.columns:
            try:
                df[col+'_num'] = pd.to_numeric(df[col], errors='coerce')
            except Exception:
                continue
        numeric_cols = [c for c in df.columns if c.endswith('_num')]

    if numeric_cols:
        numeric_field = numeric_cols[0]
        threshold = df[numeric_field].mean() if df[numeric_field].notnull().sum() else 0
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with '{numeric_field}' > {threshold:.2f} (mean): Total {len(filtered_df)} records\n")
        display(filtered_df.head())
        # Normalization
        filtered_df[numeric_field+'_normalized'] = (
            (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        )
        print(f"Normalized field '{numeric_field}':")
        display(filtered_df[[numeric_field, numeric_field+'_normalized']].head())
        # Grouping by first available non-numeric field
        group_field = None
        non_numeric_cols = [c for c in df.columns if c not in numeric_cols]
        if non_numeric_cols:
            group_field = non_numeric_cols[0]
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped mean {numeric_field} by '{group_field}':")
            display(grouped_df.head())
    else:
        print("No numeric columns found for EDA.")
else:
    print("No record set available for EDA.")

## 5. Visualization
Visualize distribution of the selected numeric field and the group-wise average, if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and numeric_cols:
    plt.figure(figsize=(10, 5))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()

    if 'grouped_df' in locals():
        plt.figure(figsize=(10, 5))
        sns.barplot(x=group_field, y=numeric_field, data=grouped_df)
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.xticks(rotation=60, ha='right')
        plt.tight_layout()
        plt.show()

## 6. Conclusion
This notebook demonstrated how to load and explore the [FAIR^2 Mass Spectrometry dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using Croissant and `mlcroissant`.

**Key findings:**
* Dataset was successfully loaded using its Croissant schema URL.
* Record sets and fields were enumerated by their `@id` ensuring robust referencing.
* Basic exploratory data analysis was performed: numeric fields were filtered, normalized, and grouped.
* Simple data visualizations were generated to illustrate the field value distribution and group-wise means.

Feel free to extend this notebook with additional feature engineering, visualizations, or machine learning models tailored to your analysis goals.